# CaseMind AI: Forensic Investigation Machine Learning Model
### 3rd Semester Machine Learning Course Project
**Objective**: Predict suspect culprit probability and case solvability based on forensic trace, alibi discrepancies, motive severity, and crime scene access vectors.

**Algorithms Benchmarked**:
1. Logistic Regression (Linear Probabilistic Baseline)
2. Decision Tree Classifier (Non-linear Rule Tree)
3. Random Forest Classifier (Bagging Ensemble)
4. Gradient Boosting Classifier (Boosting Ensemble - Champion)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

## 2. Load and Inspect Forensic Dataset

In [ ]:
# Load generated forensic benchmark dataset
df = pd.read_csv('forensic_cases.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Check class balance and missing values
print("Missing Values:\n", df.isnull().sum())
print("\nTarget Distribution (0=Innocent, 1=Culprit):\n", df['is_culprit'].value_counts(normalize=True))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.boxplot(x='is_culprit', y='alibi_gap_minutes', data=df, palette='Set2')
plt.title('Alibi Gap (Minutes) vs Culprit Status')

plt.subplot(1, 2, 2)
sns.kdeplot(data=df, x='forensic_trace_score', hue='is_culprit', common_norm=False, fill=True)
plt.title('Forensic Trace Match Score Distribution')

plt.tight_layout()
plt.show()

## 4. Feature Preprocessing & Pipeline Setup

In [ ]:
categorical_features = ['relationship_to_victim', 'motive_severity', 'access_to_crime_scene']
numerical_features = [
    'alibi_gap_minutes', 'forensic_trace_score', 'weapon_accessibility',
    'contradiction_level', 'prior_criminal_record', 'nervousness_biomarker'
]
target = 'is_culprit'

X = df[categorical_features + numerical_features]
y = df[target]

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

## 5. Model Training and Benchmarking

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=120, max_depth=9, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=5, random_state=42)
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

benchmark_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
benchmark_df.reset_index(drop=True, inplace=True)
benchmark_df

## 6. Champion Model Evaluation (Confusion Matrix & ROC Curve)

In [ ]:
champion_name = benchmark_df.iloc[0]['Model']
champion = fitted_pipelines[champion_name]
y_pred = champion.predict(X_test)
y_prob = champion.predict_proba(X_test)[:, 1]

print(f"=== Classification Report for {champion_name} ===\n")
print(classification_report(y_test, y_pred))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Innocent', 'Culprit'], yticklabels=['Innocent', 'Culprit'])
axes[0].set_title(f'Confusion Matrix - {champion_name}')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_prob):.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

In [ ]:
classifier_step = champion.named_steps['classifier']
prep_step = champion.named_steps['preprocessor']
cat_encoder = prep_step.named_transformers_['cat']
cat_feature_names = list(cat_encoder.get_feature_names_out(categorical_features))
all_feature_names = numerical_features + cat_feature_names

if hasattr(classifier_step, 'feature_importances_'):
    fi_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': classifier_step.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x='Importance', y='Feature', data=fi_df.head(10), palette='viridis')
    plt.title(f'Top 10 Most Influential Forensic Features ({champion_name})')
    plt.show()
    print(fi_df.head(10))

## 8. Export Serialized Model Pipeline

In [ ]:
# Serialize entire pipeline for FastAPI backend inference
joblib.dump(champion, 'forensic_pipeline.pkl')
print("Saved champion pipeline to 'forensic_pipeline.pkl' successfully!")